In [ ]:
# Colinearity of variables in the dataset

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("/Users/obedfrimpong/Desktop/qss-45-final-project/EJI_2024_United_States.csv")

df = df.replace(-999, np.nan)

num_df = df.select_dtypes(include=[np.number]).copy()

num_df = num_df.dropna(axis=1, how="all")
num_df = num_df.loc[:, num_df.nunique(dropna=True) > 1]

#Used
out = Path("output")
out.mkdir(exist_ok=True)

corr = num_df.corr()

corr.to_csv(out / "eji_correlation_matrix.csv")

# Find highly correlated pairs
threshold = 0.90
high_corr_pairs = []
cols = corr.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        val = corr.iloc[i, j]
        if pd.notna(val) and abs(val) >= threshold:
            high_corr_pairs.append((cols[i], cols[j], val))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=["Var1", "Var2", "Correlation"])
high_corr_df = high_corr_df.sort_values(by="Correlation", key=lambda s: s.abs(), ascending=False)
high_corr_df.to_csv(out / "high_correlation_pairs.csv", index=False)

# Heatmap of correlations
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False, cbar_kws={"shrink": 0.8})
plt.title("EJI Numeric Variable Correlation Matrix")
plt.tight_layout()
plt.savefig(out / "eji_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()

# VIF calculation
vif_df = num_df.dropna().copy()

X = sm.add_constant(vif_df)

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i) if X.columns[i] != "const" else np.nan
    for i in range(X.shape[1])
]

# Remove constant row and sort
vif_data = vif_data[vif_data["Variable"] != "const"].sort_values("VIF", ascending=False)
vif_data.to_csv(out / "vif_results.csv", index=False)

# Print summary
print("Top highly correlated pairs:")
print(high_corr_df.head(20).to_string(index=False))

print("\nTop VIF variables:")
print(vif_data.head(20).to_string(index=False))

print(f"\nSaved outputs to: {out.resolve()}")

/opt/homebrew/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/opt/homebrew/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Top highly correlated pairs:
        Var1         Var2  Correlation
       F_HVM      RPL_HVM     1.000000
  EPL_IMPWTR SPL_EBM_DOM5     1.000000
  EPL_IMPWTR RPL_EBM_DOM5     1.000000
SPL_EBM_DOM5 RPL_EBM_DOM5     1.000000
SPL_CBM_DOM1 RPL_CBM_DOM1     1.000000
    EPL_NEHD RPL_CBM_DOM1     1.000000
  EPL_MINRTY SPL_SVM_DOM1     1.000000
  EPL_MINRTY RPL_SVM_DOM1     1.000000
SPL_SVM_DOM1 RPL_SVM_DOM1     1.000000
    EPL_NEHD SPL_CBM_DOM1     1.000000
       GEOID   GEOID_2020     1.000000
     STATEFP   GEOID_2020     0.999983
     STATEFP        GEOID     0.999983
    E_HOUAGE   EPL_HOUAGE     0.996287
SPL_SVM_DOM2 RPL_SVM_DOM2     0.995177
     SPL_SER      RPL_SER     0.993641
    E_WLKIND   EPL_WLKIND    -0.993365
     SPL_SVM      RPL_SVM     0.992412
SPL_SVM_DOM4 RPL_SVM_DOM4     0.991900
      E_PARK     EPL_PARK    -0.991699

Top VIF variables:
    Variable          VIF
     STATEFP          inf
      EPL_PM 8.378790e+12
     SPL_SVM 4.642886e+12
     SPL_EBM 4.232706e+12
  